## 1) Importar bibliotecas

In [1]:
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    PowerTransformer,
    QuantileTransformer,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet

import polars as pl

## 2) Ler base de dados

In [2]:
dataset = pl.read_parquet(
    source = "../../databases/cleanned_data/diabetes_dataset.parquet"
)

print(dataset.shape)
dataset.head(2)

(442, 11)


age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32
59,2,32.099998,101.0,157.0,93.199997,38.0,4.0,4.8598,87.0,151.0
48,1,21.6,87.0,183.0,103.199997,70.0,3.0,3.8918,69.0,75.0


## 3) Aplicando processamentos

### 3.1) Pré-processamentos

In [3]:
pipeline_power = Pipeline(
    steps = [
        ("power_transformer", PowerTransformer(method = "box-cox")),
        ("poly", PolynomialFeatures(degree = 1, include_bias = False))
    ]
)

pipeline_standard = Pipeline(
    steps = [
        ("standard_scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree = 1, include_bias = False))
    ]
)

### 3.2) Column transformer

In [4]:
column_transformer = ColumnTransformer(
    transformers = [
        ("power_transform", pipeline_power, ["s4", "s5"]),
        ("standard_scaler", pipeline_standard, ["age", "bmi", "bp", "s1", "s2", "s3", "s6"]),
        ("one_hot_encoder", OneHotEncoder(), ["sex"]),
    ]
)

### 3.3) Modelos

In [5]:
model_pipeline = Pipeline(
    steps = [
        ("preprocessing", column_transformer),
        ("model", ElasticNet())
    ]
)

### 3.4) Target transformer

In [6]:
target_regressor = TransformedTargetRegressor(
    regressor = model_pipeline,
    transformer = QuantileTransformer(n_quantiles = 10)
)

target_regressor

,regressor,Pipeline(step...lasticNet())])
,transformer,QuantileTrans..._quantiles=10)
,func,None
,inverse_func,None
,check_inverse,True
,transformers,"[('power_transform', ...), ('standard_scaler', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


### 3.5) Getting paramns

In [7]:
target_regressor.get_params()

{'check_inverse': True,
 'func': None,
 'inverse_func': None,
 'regressor__memory': None,
 'regressor__steps': [('preprocessing',
   ColumnTransformer(transformers=[('power_transform',
                                    Pipeline(steps=[('power_transformer',
                                                     PowerTransformer(method='box-cox')),
                                                    ('poly',
                                                     PolynomialFeatures(degree=1,
                                                                        include_bias=False))]),
                                    ['s4', 's5']),
                                   ('standard_scaler',
                                    Pipeline(steps=[('standard_scaler',
                                                     StandardScaler()),
                                                    ('poly',
                                                     PolynomialFeatures(degree=1,
                   

## 4) Aplicando GridSearchCV

### 4.1) Construção

In [ ]:
grid_search = GridSearchCV(
    estimator = target_regressor,
    n_jobs = -1,
    cv = 5,
    scoring = "r2",
    verbose = 1,
    param_grid = {
        "regressor__model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 1],
        "regressor__model__alpha": [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "regressor__preprocessing__power_transform__power_transformer__method": ["box-cox", "yeo-johnson"],
        "regressor__preprocessing__power_transform__poly__degree": [1, 2, 3, 4, 5, 6],
        "regressor__preprocessing__standard_scaler__poly__degree": [1, 2, 3, 4, 5, 6],
        "transformer": [None, PowerTransformer(method = "box-cox"), PowerTransformer(method = "yeo-johnson")]+ [QuantileTransformer(n_quantiles = n) for n in range(5, 25, 5)],
    }
)

grid_search

,estimator,TransformedTa...quantiles=10))
,param_grid,"{'regressor__model__alpha': [0.01, 0.05, ...], 'regressor__model__l1_ratio': [0.1, 0.5, ...], 'regressor__preprocessi...transform__poly__degree': [1, 2, ...], 'regressor__preprocessi...wer_transformer__method': ['box-cox', 'yeo-johnson'], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('power_transform', ...), ('standard_scaler', ...), ...]"


### 4.2) Treinando

In [9]:
grid_search.fit(
    X = dataset.drop("target"),
    y = dataset["target"]
)

Fitting 5 folds for each of 36288 candidates, totalling 181440 fits


c:\Users\tulio\OneDrive\Área de Trabalho\Estudos\Hashtag\Ciência de dados\datascienceenv\lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [0.47953838 0.47390984 0.47392276 ... 0.03384878 0.03775489 0.0338048 ]
  warnings.warn(
c:\Users\tulio\OneDrive\Área de Trabalho\Estudos\Hashtag\Ciência de dados\datascienceenv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.097e+05, tolerance: 2.621e+02
  model = cd_fast.enet_coordinate_descent(


,estimator,TransformedTa...quantiles=10))
,param_grid,"{'regressor__model__alpha': [0.01, 0.05, ...], 'regressor__model__l1_ratio': [0.1, 0.5, ...], 'regressor__preprocessi...transform__poly__degree': [1, 2, ...], 'regressor__preprocessi...wer_transformer__method': ['box-cox', 'yeo-johnson'], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('power_transform', ...), ('standard_scaler', ...), ...]"


In [10]:
grid_search.best_params_

{'regressor__model__alpha': 0.01,
 'regressor__model__l1_ratio': 1,
 'regressor__preprocessing__power_transform__poly__degree': 2,
 'regressor__preprocessing__power_transform__power_transformer__method': 'yeo-johnson',
 'regressor__preprocessing__standard_scaler__poly__degree': 1,
 'transformer': None}

In [11]:
grid_search.best_score_

np.float64(0.4858666003212492)

In [12]:
#0.48555954550315245